# The Pacific is warming: a raw-data climate story

This notebook is the analytical source for Island Signals. It starts from the official challenge CSVs, builds the climate indicators from observations, and uses the Red List Index as one biodiversity pointer. It does **not** load the precomputed browser data or copy numbers from the application.

The central question is descriptive: what broad physical changes are visible across Pacific territories, and what does the biodiversity indicator show alongside them? These small cross-sectional data cannot, by themselves, establish that climate change caused a territory's biodiversity trend.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

DATA_DIR = Path('data/source/challenge-2026')
OUTPUT_DIR = Path('analysis/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NAME_MAP = {
    'Micronesia (Federated States of)': 'Federated States of Micronesia',
    'Micronesia, Federated State of': 'Federated States of Micronesia',
}

try:
    from scipy import stats
except ImportError:
    stats = None

def load_csv(filename):
    return pd.read_csv(DATA_DIR / filename)

def slope(years, values):
    x = np.asarray(years, dtype=float)
    y = np.asarray(values, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) < 2 or np.allclose(x, x[0]):
        return np.nan
    return float(np.polyfit(x, y, 1)[0])

def tidy_observations(filename):
    frame = load_csv(filename).copy()
    frame['name'] = frame['Pacific Island Countries and territories'].replace(NAME_MAP)
    frame['year'] = pd.to_numeric(frame['TIME_PERIOD'], errors='coerce')
    frame['value'] = pd.to_numeric(frame['OBS_VALUE'], errors='coerce')
    frame = frame[['name', 'year', 'value']].dropna().sort_values(['name', 'year'])
    duplicate_count = int(frame.duplicated(['name', 'year']).sum())
    if duplicate_count:
        raise ValueError(f'{filename} has {duplicate_count} duplicate territory-year observations')
    return frame.reset_index(drop=True)


## 1. Build comparable climate trends

A slope is a compact summary of direction and rate, not a claim that every individual year moved in a straight line. The full available period is used for the main descriptive comparison. A common 1993–2023 window is calculated too, so the notebook makes the different coverage periods visible rather than silently comparing unlike windows.

In [2]:
INDICATORS = {
    'surface temperature': ('mean-surface-temperature-anomalies.csv', 100, '°C / century'),
    'sea-surface temperature': ('mean-sea-surface-temperature-anomalies.csv', 100, '°C / century'),
    'sea level': ('sea-level-anomalies.csv', 1000, 'mm / year'),
    'rainfall': ('rainfall-anomalies.csv', 1, 'mm / year'),
}

def trend_table(label, filename, multiplier, unit):
    observations = tidy_observations(filename)
    rows = []
    for name, group in observations.groupby('name', sort=True):
        full = group.dropna(subset=['year', 'value'])
        common = full[full['year'].between(1993, 2023)]
        rows.append({
            'indicator': label,
            'territory': name,
            'unit': unit,
            'n_years': len(full),
            'start_year': int(full['year'].min()),
            'end_year': int(full['year'].max()),
            'full_slope': slope(full['year'], full['value']) * multiplier,
            'common_n_years': len(common),
            'common_slope': slope(common['year'], common['value']) * multiplier,
        })
    return pd.DataFrame(rows)

climate = pd.concat([
    trend_table(label, filename, multiplier, unit)
    for label, (filename, multiplier, unit) in INDICATORS.items()
], ignore_index=True)
climate.to_csv(OUTPUT_DIR / 'climate_story_trends.csv', index=False)

summary = (climate.groupby(['indicator', 'unit'])
           .agg(territories=('territory', 'nunique'),
                positive_full=('full_slope', lambda s: int((s > 0).sum())),
                negative_full=('full_slope', lambda s: int((s < 0).sum())),
                median_full_slope=('full_slope', 'median'),
                positive_common=('common_slope', lambda s: int((s > 0).sum())),
                negative_common=('common_slope', lambda s: int((s < 0).sum()))
           ).reset_index())
summary.round(3)

                 indicator          unit  ...  positive_common  negative_common
0                 rainfall     mm / year  ...               15                7
1                sea level     mm / year  ...               21                0
2  sea-surface temperature  °C / century  ...               21                0
3      surface temperature  °C / century  ...               22                0

[4 rows x 8 columns]

The climate spine is supported when the physical indicators show broad directional movement, while rainfall is allowed to be the exception rather than being forced into the same pattern. This distinction is important: a warm ocean, rising sea level, and warming land can be broad signals even when local rainfall trends diverge.

In [3]:
for indicator, group in climate.groupby('indicator', sort=False):
    print(
        indicator,
        '| territories:', group.territory.nunique(),
        '| full-period direction:', int((group.full_slope > 0).sum()), 'positive /', int((group.full_slope < 0).sum()), 'negative',
        '| median slope:', round(group.full_slope.median(), 3), group.unit.iloc[0],
        '| common-window direction:', int((group.common_slope > 0).sum()), 'positive /', int((group.common_slope < 0).sum()), 'negative',
    )

assert int(summary.loc[summary.indicator.eq('surface temperature'), 'positive_full'].iloc[0]) == 22
assert int(summary.loc[summary.indicator.eq('surface temperature'), 'positive_common'].iloc[0]) == 22
assert int(summary.loc[summary.indicator.eq('sea-surface temperature'), 'positive_full'].iloc[0]) == 21
assert int(summary.loc[summary.indicator.eq('sea-surface temperature'), 'positive_common'].iloc[0]) == 21
assert int(summary.loc[summary.indicator.eq('sea level'), 'positive_full'].iloc[0]) == 21
assert int(summary.loc[summary.indicator.eq('sea level'), 'positive_common'].iloc[0]) == 21
rain = summary.loc[summary.indicator.eq('rainfall')].iloc[0]
assert int(rain.positive_full) > 0 and int(rain.negative_full) > 0
print('Checks passed: warming and sea-level trends are broad; rainfall is mixed.')

surface temperature | territories: 22 | full-period direction: 22 positive / 0 negative | median slope: 0.375 °C / century | common-window direction: 22 positive / 0 negative
sea-surface temperature | territories: 21 | full-period direction: 21 positive / 0 negative | median slope: 0.372 °C / century | common-window direction: 21 positive / 0 negative
sea level | territories: 21 | full-period direction: 21 positive / 0 negative | median slope: 4.315 mm / year | common-window direction: 21 positive / 0 negative
rainfall | territories: 22 | full-period direction: 15 positive / 7 negative | median slope: 0.153 mm / year | common-window direction: 15 positive / 7 negative
Checks passed: warming and sea-level trends are broad; rainfall is mixed.


## 2. Use biodiversity as a pointer, not a causal proof

The Red List Index summarizes change in overall extinction risk; it is not a literal count of animals disappearing. The endpoint test below asks whether the 2024 score is lower than the 1993 score. That is a valid, narrow claim. It is deliberately kept separate from the stronger but incorrect statement that every territory declined every year.

In [4]:
rli_raw = load_csv('red-list-index.csv').copy()
filters = {
    'Sex': 'Total', 'Age': 'All ages', 'Urbanization': 'National',
    'Income': 'Total', 'Education level': 'All education levels',
    'Occupation': 'All occupations', 'Composite breakdown': 'Not applicable',
    'Disability': 'No breakdown by disability',
}
for column, value in filters.items():
    rli_raw = rli_raw[rli_raw[column].eq(value)]
rli_raw['territory'] = rli_raw['Pacific Island Countries and territories'].replace(NAME_MAP)
rli_raw['year'] = pd.to_numeric(rli_raw['TIME_PERIOD'], errors='coerce')
rli_raw['rli'] = pd.to_numeric(rli_raw['OBS_VALUE'], errors='coerce')
rli = rli_raw[['territory', 'year', 'rli']].dropna().sort_values(['territory', 'year'])

rows = []
for territory, group in rli.groupby('territory', sort=True):
    group = group.drop_duplicates(['territory', 'year']).sort_values('year')
    values = group.rli.to_numpy(float)
    years = group.year.to_numpy(float)
    early = group[group.year.between(1993, 2008)]
    late = group[group.year.between(2009, 2024)]
    early_slope = slope(early.year, early.rli)
    late_slope = slope(late.year, late.rli)
    rows.append({
        'territory': territory,
        'start_year': int(years.min()), 'end_year': int(years.max()),
        'start_rli': values[0], 'end_rli': values[-1],
        'endpoint_change': values[-1] - values[0],
        'annual_decreases': int((np.diff(values) < 0).sum()),
        'declined_every_year': bool((np.diff(values) < 0).all()),
        'full_slope_per_decade': slope(years, values) * 10,
        'early_slope_per_decade': early_slope * 10,
        'late_slope_per_decade': late_slope * 10,
        'acceleration_per_decade': (late_slope - early_slope) * 10,
    })

biodiversity = pd.DataFrame(rows)
biodiversity.to_csv(OUTPUT_DIR / 'biodiversity_pointer.csv', index=False)
n = len(biodiversity)
endpoint_declines = int((biodiversity.endpoint_change < 0).sum())
every_year = int(biodiversity.declined_every_year.sum())
binomial_p = sum(math.comb(n, k) for k in range(endpoint_declines, n + 1)) / (2 ** n)
faster = int((biodiversity.acceleration_per_decade < -1e-10).sum())
equal = int(np.isclose(biodiversity.acceleration_per_decade, 0, atol=1e-10).sum())
slower = n - faster - equal
print(f'RLI rows={len(rli)}, territories={n}, years={int(rli.year.min())}–{int(rli.year.max())}')
print(f'Lower endpoint: {endpoint_declines}/{n}; exact one-sided binomial p={binomial_p:.8f}')
print(f'Literal every-year decline: {every_year}/{n}')
print(f'Late decline faster / equal / slower: {faster} / {equal} / {slower}')
if stats is not None:
    paired = stats.wilcoxon(biodiversity.late_slope_per_decade, biodiversity.early_slope_per_decade, alternative='less')
    print('Wilcoxon late-vs-early signed-rank:', paired)

assert endpoint_declines == 20
assert every_year == 0
assert (faster, equal, slower) == (18, 1, 3)
biodiversity.sort_values('endpoint_change').head()

RLI rows=704, territories=22, years=1993–2024
Lower endpoint: 20/22; exact one-sided binomial p=0.00006056
Literal every-year decline: 0/22
Late decline faster / equal / slower: 18 / 1 / 3


                         territory  ...  acceleration_per_decade
5                             Guam  ...                 0.006324
12                           Palau  ...                 0.003676
14                        Pitcairn  ...                -0.003529
2   Federated States of Micronesia  ...                -0.009853
11        Northern Mariana Islands  ...                 0.012941

[5 rows x 12 columns]

## 3. Optional association check

A climate-first story does not need a territory-by-territory causal correlation. This check is retained as a guardrail: it shows what the small dataset can and cannot say. The correlations below compare full-period climate slopes with the Red List trend across territories, so they are exploratory associations with different measurement windows and many unmeasured confounders.

In [5]:
def correlation_row(indicator, climate_values):
    merged = biodiversity[['territory', 'full_slope_per_decade']].merge(
        climate_values[['territory', 'full_slope']], on='territory', how='inner'
    ).dropna()
    x, y = merged.full_slope_per_decade.to_numpy(), merged.full_slope.to_numpy()
    result = {'indicator': indicator, 'n': len(merged), 'pearson_r': np.corrcoef(x, y)[0, 1]}
    if stats is not None:
        result['pearson_p'] = stats.pearsonr(x, y).pvalue
        result['spearman_rho'] = stats.spearmanr(x, y).statistic
        result['spearman_p'] = stats.spearmanr(x, y).pvalue
    return result

associations = pd.DataFrame([
    correlation_row(indicator, group)
    for indicator, group in climate.groupby('indicator', sort=False)
])
associations.round(3)

                 indicator   n  pearson_r
0      surface temperature  22      0.215
1  sea-surface temperature  21      0.215
2                sea level  21      0.117
3                 rainfall  22     -0.017

## Decision for the story

Use the direct climate indicators as the main narrative spine: warming land, warming sea surface, rising sea level, and locally mixed rainfall. Then use the Red List Index as one meaningful pointer to ecological pressure. This is a stronger fit for the competition's climate-change theme and is more honest about what the data supports. The story should say **alongside**, **co-occurs with**, or **points to**, not that these CSVs prove climate change caused the biodiversity trend.